# LSTM CARAVAN — Google Colab (full-scale replication)

Colab-adapted version of `LSTM_CARAVAN.ipynb` that reproduces the full 17,847-basin Caravan pretraining run from **[examples/configs/caravan.yml](../examples/configs/caravan.yml)** without needing the pre-existing 173GB `data/Caravan_all` folder on the cluster.

**How this differs from the cluster setup:**
- The full Caravan dataset isn't one download — it's the core Caravan release plus 9 separately-hosted community extensions (camelsch, camelscz, camelsde, camelsdk, camelses, GRDC, il, ausvic, lamahice). This notebook downloads all of them directly from their original Zenodo/HydroShare records (cloud-to-cloud, much faster than uploading from your own connection) and merges them into the same `attributes/<source>/` + `timeseries/csv/<source>/` layout Hy2DL expects. Every URL below was verified by matching its exact byte size against the dataset already staged on the cluster.
- The assembled dataset is downloaded from source **once**, then archived to Google Drive (`/content/drive/MyDrive/Hy2DL_Caravan_data/Caravan_all.tar`, ~137GB). Every session after the first restores it from that single Drive archive instead of re-downloading from Zenodo/HydroShare — a bulk copy of one big file is much faster than repeating 9 separate downloads + extraction.
- Model checkpoints, logs, and the processed dataset cache are also written to Drive (`/content/drive/MyDrive/Hy2DL_Caravan_results`), so training survives a disconnect — rerunning the training cell resumes from the last completed epoch instead of starting over.
- One caveat: **LamaH-Ice** (74 of 17,847 basins, 0.4%) is hosted on HydroShare, which doesn't version files the way Zenodo does. The copy available there today doesn't byte-match the one used to build the cluster's 173GB dataset — it's very likely just a content update on their end, not a broken link, but flagged separately below so you can decide whether to include it.

**Expect:** ~41GB download and ~137GB of Drive usage on the first run only; later sessions just restore from Drive. ~90h of training total (30 epochs) — plan on multiple Colab sessions, with the resume logic below carrying you across disconnects. Recommended: Colab Pro/Pro+ with an A100/L4 GPU, and enough free Drive space (~140GB) for the cached dataset plus results.

## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/sanikabaste/Hy2DL_new.git
%cd /content/Hy2DL_new
!pip install -q -e .

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Download and assemble the Caravan dataset

First checks Google Drive for a cached archive from a previous session. If found, restores it directly (fast) and the download/extraction cells below skip themselves. Otherwise downloads, extracts, and merges each source in turn (rather than all at once) to keep peak local disk usage bounded, then archives the assembled dataset to Drive at the end so future sessions can restore instead of re-downloading.

In [ ]:
import shutil
import subprocess
from pathlib import Path

CARAVAN_DIR = Path("data/Caravan_all").resolve()
DRIVE_CACHE_DIR = Path("/content/drive/MyDrive/Hy2DL_Caravan_data")
DRIVE_CACHE_TAR = DRIVE_CACHE_DIR / "Caravan_all.tar"

if DRIVE_CACHE_TAR.exists():
    size_gb = DRIVE_CACHE_TAR.stat().st_size / 1e9
    print(f"Found cached dataset on Drive ({size_gb:.1f} GB). Restoring locally instead of re-downloading...")
    CARAVAN_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xf", str(DRIVE_CACHE_TAR), "-C", str(CARAVAN_DIR.parent)], check=True)
    NEED_DOWNLOAD = False
    print("Restored from Drive cache. The download/extraction/caching cells below will skip themselves.")
else:
    print("No Drive cache found — this must be the first run. Will download from source and cache to Drive at the end.")
    (CARAVAN_DIR / "attributes").mkdir(parents=True, exist_ok=True)
    (CARAVAN_DIR / "timeseries" / "csv").mkdir(parents=True, exist_ok=True)
    NEED_DOWNLOAD = True

In [ ]:
SCRATCH = Path("/content/_caravan_scratch")

# (name, url, archive filename, exact expected size in bytes, archive kind)
# Every URL/size pair was verified against the dataset already staged on the cluster before being added here.
ARCHIVES = [
    (
        "Core Caravan (camels, camelsaus, camelsbr, camelscl, camelsgb, hysets, lamah)",
        "https://zenodo.org/api/records/10968468/files/Caravan-csv.tar.xz/content",
        "Caravan-csv.tar.xz",
        23433742084,
        "tar",
    ),
    (
        "GRDC extension",
        "https://zenodo.org/api/records/15349031/files/GRDC_Caravan_extension_csv.zip/content",
        "GRDC_Caravan_extension_csv.zip",
        8842746230,
        "zip",
    ),
    (
        "CAMELS-DE extension",
        "https://zenodo.org/api/records/14755229/files/caravan_de.zip/content",
        "caravan_de.zip",
        6085830462,
        "zip",
    ),
    (
        "CAMELS-CH extension",
        "https://zenodo.org/api/records/15025258/files/Caravan_extension_CH.zip/content",
        "Caravan_extension_CH.zip",
        545597282,
        "zip",
    ),
    (
        "CAMELS-CZ extension",
        "https://zenodo.org/api/records/17769325/files/Caravan-Extension-CZ.zip/content",
        "Caravan-Extension-CZ.zip",
        913172886,
        "zip",
    ),
    (
        "Denmark extension",
        "https://zenodo.org/api/records/15200118/files/Caravan_extension_DK.zip/content",
        "Caravan_extension_DK.zip",
        521600377,
        "zip",
    ),
    (
        "CAMELS-ES extension",
        "https://zenodo.org/api/records/15040948/files/CAMELS-ES_v110.zip/content",
        "CAMELS-ES_v110.zip",
        402788602,
        "zip",
    ),
    (
        "Israel extension",
        "https://zenodo.org/api/records/15181680/files/Caravan_extension_Israel_Ver4.zip/content",
        "Caravan_extension_Israel_Ver4.zip",
        294290689,
        "zip",
    ),
    (
        "Australia-Victoria extension",
        "https://zenodo.org/api/records/20627056/files/caravan_ausvic.zip/content",
        "caravan_ausvic.zip",
        33652672,
        "zip",
    ),
]


def merge_extracted(extract_root: Path, target_root: Path) -> list[str]:
    """Archives wrap their content in a varying number of nested folders. Find every 'attributes'
    directory and every 'timeseries/csv' directory anywhere inside extract_root and move each
    source subfolder into the matching canonical location under target_root; everything else
    (shapefiles, licenses, netcdf, docs) is left behind and discarded with the scratch directory.
    """
    moved = []
    for attributes_dir in extract_root.rglob("attributes"):
        if attributes_dir.is_dir():
            for source_dir in attributes_dir.iterdir():
                if source_dir.is_dir():
                    dest = target_root / "attributes" / source_dir.name
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(source_dir), str(dest))
                    moved.append(f"attributes/{source_dir.name}")
    for csv_dir in extract_root.rglob("csv"):
        if csv_dir.is_dir() and csv_dir.parent.name == "timeseries":
            for source_dir in csv_dir.iterdir():
                if source_dir.is_dir():
                    dest = target_root / "timeseries" / "csv" / source_dir.name
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(source_dir), str(dest))
                    moved.append(f"timeseries/csv/{source_dir.name}")
    return moved


def process_archive(name: str, url: str, filename: str, expected_size: int, kind: str) -> None:
    print(f"=== {name} ===")
    archive_path = SCRATCH / filename
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(archive_path), url], check=True)

    actual_size = archive_path.stat().st_size
    if actual_size != expected_size:
        raise RuntimeError(
            f"{filename}: downloaded {actual_size} bytes, expected {expected_size}. "
            "The upstream file may have changed — stopping rather than risk training on the wrong data."
        )

    extract_dir = SCRATCH / "extracted"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir()

    print(f"  Extracting {filename} ...")
    if kind == "tar":
        subprocess.run(["tar", "-xJf", str(archive_path), "-C", str(extract_dir)], check=True)
    else:
        subprocess.run(["unzip", "-q", str(archive_path), "-d", str(extract_dir)], check=True)

    moved = merge_extracted(extract_dir, CARAVAN_DIR)
    print(f"  Merged {len(moved)} source folder(s): {moved}")

    archive_path.unlink()
    shutil.rmtree(extract_dir)
    print(f"  Done with {name}.\n")


if NEED_DOWNLOAD:
    SCRATCH.mkdir(exist_ok=True)
    for args in ARCHIVES:
        process_archive(*args)
    print("Core Caravan + all 8 verified extensions processed.")
else:
    print("Skipping download — using dataset restored from Drive cache.")

### LamaH-Ice (optional — 74 basins, 0.4% of the pretrain set)

Hosted on HydroShare rather than Zenodo. The version available there today (283,091,076 bytes) does **not** byte-match the copy used to build the cluster's 173GB dataset (383,647,089 bytes) — HydroShare doesn't version files the way Zenodo does, so this is very likely a content update since the cluster copy was fetched, not a broken link. Run the cell below to include it anyway (basin set/values for Iceland may differ slightly from the cluster run), or skip it to train on the other 17,773 basins only. Only relevant on the first run — whatever you decide here becomes part of the Drive cache for every session after.

In [ ]:
if NEED_DOWNLOAD:
    process_archive(
        "LamaH-Ice extension",
        "https://www.hydroshare.org/resource/86117a5f36cc4b7c90a5d54e18161c91/data/contents/Caravan_extension_lamahice.zip/",
        "Caravan_extension_lamahice.zip",
        283091076,
        "zip",
    )
else:
    print("Skipping — using dataset restored from Drive cache (already includes/excludes LamaH-Ice per your first-run choice).")

In [ ]:
# Sanity check: per-source basin counts, and how many basins from the pretrain list are actually present.
print(f"{'source':15s} {'basins':>8s}")
total = 0
for d in sorted((CARAVAN_DIR / "timeseries" / "csv").iterdir()):
    if d.is_dir():
        n = len(list(d.glob("*.csv")))
        total += n
        print(f"{d.name:15s} {n:8d}")
print(f"{'TOTAL':15s} {total:8d}")

basin_list = set(Path("data/basin_id/basins_caravan_pretrain.txt").read_text().split())
downloaded = {p.stem for p in (CARAVAN_DIR / "timeseries" / "csv").rglob("*.csv")}
print(f"\nPretrain basin list: {len(basin_list)} basins")
print(f"Matched: {len(basin_list & downloaded)}")
missing = basin_list - downloaded
print(f"Missing: {len(missing)}")
if missing:
    print("Sample missing:", list(missing)[:10])

In [ ]:
# Cache the assembled dataset to Drive so future sessions can restore instead of re-downloading.
if NEED_DOWNLOAD:
    DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Archiving {CARAVAN_DIR} to {DRIVE_CACHE_TAR} (one-time cost, streams straight to Drive)...")
    subprocess.run(
        ["tar", "-cf", str(DRIVE_CACHE_TAR), "-C", str(CARAVAN_DIR.parent), CARAVAN_DIR.name],
        check=True,
    )
    print(f"Cached {DRIVE_CACHE_TAR.stat().st_size / 1e9:.1f} GB to Drive. Future sessions will restore from here.")
else:
    print("Already restored from an existing Drive cache — nothing to save.")

## 3. Configure the experiment

Same `caravan.yml` as the cluster run, with `path_save_folder` redirected to Google Drive so checkpoints/logs survive a Colab disconnect, and the processed training/validation datasets cached there too (skips the lengthy dataset-processing step on a resumed session).

In [ ]:
import datetime
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr
import yaml

from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation import calculate_metrics, get_tester
from hy2dl.modelzoo import get_model
from hy2dl.training.basetrainer import BaseTrainer
from hy2dl.utils.config import Config

base_dir = Path.cwd().resolve()
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

with open("examples/configs/caravan.yml") as f:
    cfg_dict = yaml.safe_load(f)

cfg_dict["path_save_folder"] = "/content/drive/MyDrive/Hy2DL_Caravan_results"

config = Config(cfg_dict, base_dir=base_dir)
config.init_experiment()

config.path_dataset_training = config.path_save_folder / "dataset_training.zarr"
config.path_dataset_validation = config.path_save_folder / "dataset_validation.zarr"

config.dump()

Dataset = get_dataset(config)
Tester = get_tester(config)

In [ ]:
# Create training dataset
training_dataset = Dataset(cfg=config, time_period="training")
training_dataset.setup_dataset()
# Initialize training object
trainer = BaseTrainer(cfg=config, training_dataset=training_dataset)

In [ ]:
validation_dataset = Dataset(cfg=config, time_period="validation")
validation_dataset.setup_dataset(check_nan=False, path_scaler=config.path_save_folder / "scaler.yml")
tester_validation = Tester(cfg=config, evaluation_dataset=validation_dataset)

## 4. Train (resumable)

**If your Colab session disconnects:** reconnect, re-run every cell above (repo clone/install, Drive mount, dataset restore-from-cache, config/dataset setup), then re-run this cell — it detects the last checkpoint saved on Drive and continues from `last_epoch + 1` instead of restarting.

In [ ]:
model_dir = config.path_save_folder / "model"
completed_epochs = sorted(int(p.name.rsplit("_", 1)[1]) for p in model_dir.glob("model_epoch_*"))
start_epoch = 1
if completed_epochs:
    last_epoch = completed_epochs[-1]
    trainer.model.load_state_dict(
        torch.load(model_dir / f"model_epoch_{last_epoch}", map_location=config.device)
    )
    trainer.optimizer.update_optimizer_lr(epoch=last_epoch + 1)
    start_epoch = last_epoch + 1
    print(f"Resuming training from epoch {start_epoch} (found checkpoint for epoch {last_epoch}).")

if start_epoch > config.epochs:
    print(f"All {config.epochs} epochs already completed.")
else:
    validation_headers = "".join([f"{m:^10}|" for m in config.validation_metric])
    config.logger.info("Training model".center(60, "-"))
    config.logger.info(f"{'':^16}|{'Training':^21}|{'Validation':^{(11 * len(config.validation_metric)) + 10}}|")
    config.logger.info(f"{'Epoch':^5}|{'LR':^10}|{'Loss':^10}|{'Time':^10}|{validation_headers}{'Time':^10}|")

    total_time = time.time()
    for epoch in range(start_epoch, config.epochs + 1):
        trainer.train_model(epoch=epoch)  # Training
        tester_validation.validate_model(model=trainer.model, epoch=epoch)  # Validation
        config.logger.info(trainer.report + tester_validation.validation_report)  # report

    config.logger.info(f"Total training time: {datetime.timedelta(seconds=int(time.time() - total_time))}\n")
    shutil.rmtree(tester_validation.path_zarr, ignore_errors=True)  # delete validation results

## 5. Test and evaluate

In [ ]:
# Reconstruct the model from the last saved epoch and evaluate it on the testing period
model = get_model(config).to(config.device)
model.load_state_dict(
    torch.load(config.path_save_folder / "model" / f"model_epoch_{config.epochs}", map_location=config.device)
)

testing_dataset = Dataset(cfg=config, time_period="testing")
testing_dataset.setup_dataset(check_nan=False, path_scaler=config.path_save_folder / "scaler.yml")
tester_testing = Tester(cfg=config, evaluation_dataset=testing_dataset)

config.logger.info("Testing model...")
testing_time = time.time()
tester_testing.evaluate_model(model=model)
config.logger.info("Testing completed.")
config.logger.info(f"Total testing time: {datetime.timedelta(seconds=int(time.time() - testing_time))}\n")

test_results = xr.open_zarr(tester_testing.path_zarr)
testing_metrics = calculate_metrics(ds_results=test_results, metric_name=config.testing_metrics)
testing_metrics.to_zarr(config.path_save_folder / "testing_metrics.zarr", mode="w")

In [ ]:
# Loss testing
target_of_interest = random.sample(list(testing_metrics.feature.values), 1)[0]
test_metric = testing_metrics.sel(feature=target_of_interest, metric="nse").round(3).T.to_pandas().dropna()
# Plot the histogram
plt.figure(figsize=(10, 5))
plt.hist(test_metric, bins=np.linspace(0.0, 1.0, 11).tolist())
# Add NSE statistics to the plot
plt.text(
    0.01,
    0.8,
    (
        f"Mean: {'%.2f' % test_metric.mean():>7}\n"
        f"Median: {'%.2f' % test_metric.median():>0}\n"
        f"Max: {'%.2f' % test_metric.max():>9}\n"
        f"Min: {'%.2f' % test_metric.min():>10}"
    ),
    transform=plt.gca().transAxes,
    bbox=dict(facecolor="white", alpha=0.5),
)

# Format plot
plt.xlabel("NSE", fontsize=12, fontweight="bold")
plt.ylabel("Frequency", fontsize=12, fontweight="bold")
plt.title(f"NSE histogram for: {target_of_interest}", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot simulated and observed discharges
basin_to_analyze = random.sample(list(test_results.gauge_id.values), 1)[0]
y_sim = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_sim"].compute().values
y_obs = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_obs"].compute().values

plt.figure(figsize=(15, 7.5))
plt.plot(y_obs, label="observed", color=color_palette["observed"])
plt.plot(y_sim, label="simulated", alpha=0.5, color=color_palette["simulated"])

# Format plot
plt.xlabel("Date", fontsize=12, fontweight="bold")
plt.ylabel(target_of_interest, fontsize=12, fontweight="bold")
plt.title(f"Results for gauge_id: {basin_to_analyze}", fontsize=16, fontweight="bold")
plt.tick_params(axis="both", which="major", labelsize=12)
plt.legend(loc="upper right", fontsize=12)
plt.tight_layout()
plt.show()